### Tools

Models can request to call tools that performs task such as fetching data from a database, searching the web, or running code. Tools are pairing of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots don’t “talk” in the human sense of using language intentionally; they are exceptional mimics that can reproduce sounds they hear, including human speech. Their ability to do this stems from a combination of anatomy, brain structure, social habits, and evolutionary pressures.\n\n### 1. Anatomical and neurological basis\n| Feature | Why it matters for mimicry |\n|---------|----------------------------|\n| **Syrinx** (the bird’s vocal organ) | Located at the base of the trachea, the syrinx has highly flexible membranes and muscles that can produce a wide range of tones, far more varied than the human larynx. |\n| **Brain regions** (especially the **song system**) | Parrots have an enlarged “core” of the forebrain (the **nidopallium caudolaterale**, analogous to the mammalian prefrontal cortex) that supports complex vocal learning and flexible sound production. |\n| **Auditory memory** | They can store and recall sounds with great fidelity, a prerequisite for acc

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)-> str:
    """Get the weather of a city."""
    return f"The weather of {location} is Rainy."

model_with_tools=model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather in Ghaziabad?")
response

AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks for weather in Ghaziabad. Need to call function get_weather with location "Ghaziabad".', 'tool_calls': [{'id': 'fc_6dfe70cb-1a4b-410b-bfc2-04f9a8674502', 'function': {'arguments': '{"location":"Ghaziabad"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 128, 'total_tokens': 180, 'completion_time': 0.109608019, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.01266766, 'prompt_tokens_details': None, 'queue_time': 0.357090105, 'total_time': 0.122275679}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_02b0d31eca', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0eda2-58de-7ad0-a534-085943e7677c-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Ghaziabad'}, 'id': 'fc_6dfe70cb-1a4b-410b-bfc2-04f9a8674502', 'type': 'tool_c

### Tool Execution Loop

In [4]:
#Step 1: Model generates tool calls
messages = [{"role":"user", "content":"What's the weather like in Ghaziabad"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#Step 2: Execute tools and collect results
for tool_call in  ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#Step 3:Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The current weather in Ghaziabad is **rainy**. 🌧️ If you need a forecast for later today or any other details (like temperature, humidity, or rain‑chance), just let me know!



```python
messages = [{"role": "user", "content": "What's the weather like in Ghaziabad"}]
```

`messages` is a variable name—a labeled box where Python stores something.

`=` means “store the thing on the right inside the name on the left.” It does not mean “equals” in the maths sense here.

The value on the right is:

```python
[{"role": "user", "content": "What's the weather like in Ghaziabad"}]
```

`[ ]` creates a list: an ordered collection of things.

Inside it is:

```python
{"role": "user", "content": "What's the weather like in Ghaziabad"}
```

`{ }` creates a dictionary: a collection of labeled values.

- `"role"` is a label, or key.
- `"user"` is the value for that label.
- `"content"` is another label.
- The question is its value.

So, in plain English: “Create a conversation history containing one message—a message from the user asking about Ghaziabad’s weather.”

---

```python
ai_msg = model_with_tools.invoke(messages)
```

Again, `ai_msg` is a variable name and `=` stores the result.

`model_with_tools` is your LLM after you have attached the `get_weather` tool to it.

`.` means “use something that belongs to this object.”

So:

```python
model_with_tools.invoke
```

means: “use the `invoke` action provided by `model_with_tools`.”

`(messages)` passes `messages` into that action.

In plain English:

> Send the conversation history to the LLM and store its next response in `ai_msg`.

At this stage, the model might respond normally, or it might produce a tool request. For this prompt, it may produce something conceptually like:

```python
AI: Call get_weather with location="Ghaziabad"
```

That request is stored inside `ai_msg.tool_calls`.

---

```python
messages.append(ai_msg)
```

`.append(...)` means “add one item to the end of a list.”

So this adds the LLM’s response—its tool request—to the conversation history.

The history is now roughly:

```text
1. User: What’s the weather like in Ghaziabad?
2. AI: I want to call get_weather for Ghaziabad.
```

---

```python
for tool_call in ai_msg.tool_calls:
```

`for` starts a loop: “repeat some code.”

`ai_msg.tool_calls` means the list of tool requests that the LLM made.

`in` means “go through each item in.”

`tool_call` is a temporary variable name. On every repetition, it stores one tool request.

For example, if the LLM asked for weather in Ghaziabad and Delhi, then Python would run the indented code twice:

```text
First time: tool_call = request for Ghaziabad
Second time: tool_call = request for Delhi
```

The `:` means “the indented lines underneath belong to this loop.”

---

```python
    tool_result = get_weather.invoke(tool_call)
```

The spaces at the beginning are important. They tell Python this line is part of the `for` loop.

`get_weather` is your tool.

`.invoke(tool_call)` means:

> Run `get_weather` using the arguments contained in this tool request.

If the tool request contains:

```python
{"location": "Ghaziabad"}
```

then Python effectively runs the equivalent of:

```python
get_weather(location="Ghaziabad")
```

Your function returns:

```python
"The weather in Ghaziabad is rainy."
```

That returned text is stored in `tool_result`.

---

```python
    messages.append(tool_result)
```

This adds the tool’s answer to the end of `messages`.

Now the history is:

```text
1. User: What’s the weather like in Ghaziabad?
2. AI: I want to call get_weather for Ghaziabad.
3. Tool: The weather in Ghaziabad is rainy.
```

---

```python
final_response = model_with_tools.invoke(messages)
```

This sends the updated history to the LLM again.

The model can now see:

- what the user asked,
- that it requested the weather tool,
- and the tool’s actual result.

It then creates its next response. Usually, this is a friendly answer for the user:

```text
The weather in Ghaziabad is rainy.
```

That new AI message is saved in `final_response`.

---

```python
print(final_response.text)
```

`print(...)` displays something in the notebook output.

`final_response.text` means “get the text part of the response.”

So this prints only the final readable answer, rather than showing all the extra technical information inside `final_response`.

The full process is:

```text
User message
    ↓
LLM decides whether it needs a tool
    ↓
Python runs the requested tool
    ↓
Tool result is added to the history
    ↓
LLM reads the result and writes the final answer
```

One subtle point: the Python `for` loop does not decide which tool to use. The LLM decides that. The loop’s job is only to run every tool request the LLM has made.